# Recurrent Neural Networks: A Small Hands-On Study

This notebook demonstrates sequence handling, a basic recurrent cell, token encoding, vanishing gradients, and a short backpropagation-through-time example.

## 1. A sequence of daily activity values

In [ ]:

activity = [5400, 6800, 7350, 6100, 8200, 7900, 9050]

for day_number, count in enumerate(activity, start=1):
    print(f"Day {day_number}: {count:,} steps")


## 2. Implementing a scalar RNN cell

The hidden state is updated using a `tanh` activation. The recurrent matrix carries information from the previous time step.

In [ ]:

import numpy as np

signal = [95, 130, 175, 155, 215]

input_weight = np.array([[0.012]])
recurrent_weight = np.array([[0.45]])
bias = np.zeros((1, 1))
hidden = np.zeros((1, 1))

for value in signal:
    current_input = np.array([[value]], dtype=float)
    hidden = np.tanh(
        input_weight @ current_input
        + recurrent_weight @ hidden
        + bias
    )

print("Hidden representation:", hidden)


## 3. Encoding a sentence as integers

In [ ]:

vocabulary = {
    "we": 1,
    "enjoy": 2,
    "watching": 3,
    "the": 4,
    "final": 5,
    "together": 6,
}

tokens = ["we", "enjoy", "watching", "the", "final", "together"]
encoded_tokens = [vocabulary[token] for token in tokens]

print("Tokens:", tokens)
print("Encoded sequence:", encoded_tokens)


In [ ]:

input_matrix = np.array([[0.16]])
memory_matrix = np.array([[0.58]])
offset = np.array([[0.02]])
state = np.zeros((1, 1))

for token in tokens:
    token_value = np.array([[vocabulary[token]]], dtype=float)
    state = np.tanh(input_matrix @ token_value + memory_matrix @ state + offset)
    print(f"{token:10s} -> state={state.item():.5f}")


## 4. Why gradients can vanish

In a basic RNN, gradients are passed backward through many recurrent steps. If the repeated derivatives are smaller than one, their product can become very small. Earlier inputs then have little influence on the parameter updates.

For example, when processing a long sentence, a basic RNN may struggle to preserve the subject information needed to choose the correct verb. LSTM and GRU architectures introduce gates that help preserve useful information across longer sequences.

## 5. A compact BPTT calculation

The following example performs a forward pass over three time steps and accumulates the recurrent-weight gradient in reverse order.

In [ ]:

import numpy as np

W_input = np.array([[0.42]])
W_memory = np.array([[0.72]])
W_output = np.array([[1.10]])

inputs = [np.array([[0.8]]), np.array([[1.6]]), np.array([[2.4]])]
targets = [np.array([[0.7]]), np.array([[1.5]]), np.array([[2.2]])]

states = [np.zeros((1, 1))]
outputs = []

# Forward propagation
for step, value in enumerate(inputs):
    next_state = np.tanh(W_input @ value + W_memory @ states[step])
    prediction = W_output @ next_state

    states.append(next_state)
    outputs.append(prediction)

# Backward propagation through time
gradient_memory = np.zeros_like(W_memory)
carry_gradient = np.zeros_like(states[0])

for step in range(len(inputs) - 1, -1, -1):
    output_error = outputs[step] - targets[step]
    state_gradient = W_output.T @ output_error + carry_gradient
    activation_gradient = state_gradient * (1 - states[step + 1] ** 2)

    gradient_memory += activation_gradient @ states[step].T
    carry_gradient = W_memory.T @ activation_gradient

print("Forward hidden states:")
for index, state_value in enumerate(states[1:], start=1):
    print(index, state_value.ravel())

print("\nAccumulated recurrent-weight gradient:")
print(gradient_memory)


### Key takeaways

- A recurrent state summarizes earlier inputs.
- Token IDs can be processed sequentially, although practical NLP systems usually use embeddings.
- Vanishing gradients make long-range learning difficult in basic RNNs.
- BPTT applies the chain rule backward across time steps.